## KERAS for CITEseq & Multiome
- This dataset comprises single-cell multiomics data collected from mobilized peripheral CD34+ hematopoietic stem and progenitor cells (HSPCs) isolated from four healthy human donors. 
- From each culture plate at each sampling time point, cells were collected for measurement with two single-cell assays. The first is the 10x Chromium Single Cell Multiome ATAC + Gene Expression technology (Multiome) and the second is the 10x Genomics Single Cell Gene Expression with Feature Barcoding technology technology using the TotalSeq™-B Human Universal Cocktail, V1.0 (CITEseq).

### Data
- Multiome
    - train/test_multi_inputs.h5 - ATAC-seq peak counts transformed with TF-IDF using the default log(TF) * log(IDF) output (chromatin accessibility), with rows corresponding to cells and columns corresponding to the location of the genome whose level of accessibility is measured, here identified by the genomic coordinates on reference genome GRCh38 provided in the 10x References - 2020-A (July 7, 2020).
    - train_multi_targets.h5 - RNA gene expression levels as library-size normalized and log1p transformed counts for the same cells.
- CITEseq
    - train/test_cite_inputs.h5 - RNA library-size normalized and log1p transformed counts (gene expression levels), with rows corresponding to cells and columns corresponding to genes given by {gene_name}_{gene_ensemble-ids}.
    - train_cite_targets.h5 - Surface protein levels for the same cells that have been dsb normalized.

### ML question framing
- For Multiome samples: predict gene expression from chromatin accessibility.
- For CITE-seq samples: predict protein levels from gene expression.

In [ ]:
# Standard library
import os
import gc
import warnings
from typing import Union, Optional, Tuple, List, Dict, Any
from pathlib import Path
import pickle

# Type definitions
PathLike = Union[str, Path]


# Core data science
import numpy as np
import pandas as pd
import h5py
import scipy

# Scikit-learn
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")

DATA_DIR = Path("/share/crsp/lab/pkaiser/ddlin/single-cell-multimodal-ml/data")
RAW_DIR = DATA_DIR.joinpath("raw")
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FP_CITE_TRAIN_INPUTS  = RAW_DIR.joinpath("train_cite_inputs.h5")
FP_CITE_TRAIN_TARGETS = RAW_DIR.joinpath("train_cite_targets.h5")
FP_CITE_TEST_INPUTS   = RAW_DIR.joinpath("test_cite_inputs.h5")

FP_MULTIOME_TRAIN_INPUT = RAW_DIR.joinpath("train_multi_inputs.h5")
FP_MULTIOME_TRAIN_TARGETS = RAW_DIR.joinpath("train_multi_targets.h5")
FP_MULTIOME_TEST_INPUTS   = RAW_DIR.joinpath("test_multi_inputs.h5")

FP_SUBMISSION         = RAW_DIR.joinpath("sample_submission.csv")
FP_EVALUATION_IDS     = RAW_DIR.joinpath("evaluation_ids.csv")
FP_CELL_METADATA      = RAW_DIR.joinpath("metadata.csv")

VERBOSE = 0

## ------ CITEseq MODEL ---------

### Important cell surface proteins

We now define two sets of features:
- `constant_cols` is the set of all features which are constant in the train or test datset. These columns will be discarded immediately after loading.
- `important_cols` is the set of all features whose name matches the name of a target protein. If a gene is named 'ENSG00000114013_CD86', it should be related to a protein named 'CD86'. These features will be used for the model unchanged, that is, they don't undergo dimensionality reduction. 

In [6]:
# Curated constant and important columns from dimension reduction
constant_path   = DATA_DIR.joinpath("processed", "constant_cols.txt")
important_path  = DATA_DIR.joinpath("processed", "important_cols.txt")

with open(constant_path, 'r') as f:
    constant_cols = [line.strip() for line in f]

with open(important_path, 'r') as f:
    important_cols = [line.strip() for line in f]

# Load selected cols and print the first 5 elements
print(f'Loaded {len(constant_cols)} constant columns')
print(f'First 5 constant columns: {constant_cols[:5]}')

print(f'Loaded {len(important_cols)} important columns')
print(f'First 5 important columns: {important_cols[:5]}')

Loaded 1194 constant columns
First 5 constant columns: ['ENSG00000003137_CYP26B1', 'ENSG00000004848_ARX', 'ENSG00000006606_CCL26', 'ENSG00000010379_SLC6A13', 'ENSG00000010932_FMO1']
Loaded 84 important columns
First 5 important columns: ['ENSG00000135218_CD36', 'ENSG00000010278_CD9', 'ENSG00000204287_HLA-DRA', 'ENSG00000117091_CD48', 'ENSG00000004468_CD38']


### Preparation of the cross validation by donors and sparse matrix creation

In [7]:
metadata_df = pd.read_csv(FP_CELL_METADATA, index_col = 'cell_id')

# Filter metadata to only include CITE-seq technology
metadata_df = metadata_df[metadata_df.technology == "citeseq"]

# Check out the metadata
print(f'Metadata shape {metadata_df.shape}')
metadata_df.head()


Metadata shape (119651, 4)


,day,donor,cell_type,technology
cell_id,,,,
c2150f55becb,2,27678,HSC,citeseq
65b7edf8a4da,2,27678,HSC,citeseq
c1b26cb1057b,2,27678,EryP,citeseq
917168fa6f83,2,27678,NeuP,citeseq
2b29feeca86d,2,27678,EryP,citeseq


In [8]:
# Read train and convert to sparse matrix, we dont' want to load the constant columns since they don't have any information
print(f'Reading CITE-seq train inputs from {FP_CITE_TRAIN_INPUTS}')
X = pd.read_hdf(FP_CITE_TRAIN_INPUTS).drop(columns = constant_cols)

# Reindex and filter metadata to match the cell index in X
cell_index = X.index
meta = metadata_df.reindex(cell_index)

# Take a look of X, this is a huge file, we will later clean it up to save memory
X.head()


Reading CITE-seq train inputs from /share/crsp/lab/pkaiser/ddlin/single-cell-multimodal-ml/data/raw/train_cite_inputs.h5


gene_id,ENSG00000121410_A1BG,ENSG00000268895_A1BG-AS1,ENSG00000175899_A2M,ENSG00000245105_A2M-AS1,ENSG00000128274_A4GALT,ENSG00000094914_AAAS,ENSG00000081760_AACS,ENSG00000109576_AADAT,ENSG00000103591_AAGAB,ENSG00000115977_AAK1,...,ENSG00000153975_ZUP1,ENSG00000086827_ZW10,ENSG00000174442_ZWILCH,ENSG00000122952_ZWINT,ENSG00000198205_ZXDA,ENSG00000198455_ZXDB,ENSG00000070476_ZXDC,ENSG00000162378_ZYG11B,ENSG00000159840_ZYX,ENSG00000074755_ZZEF1
cell_id,,,,,,,,,,,,,,,,,,,,,
45006fe3e4c8,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.00000,0.000000,4.090185,0.0
d02759a80ba2,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,4.039545,...,0.000000,0.000000,0.000000,4.039545,0.0,0.0,0.00000,0.000000,0.000000,0.0
c016c6b0efa5,0.0,0.0,0.0,0.0,3.847321,0.000000,3.847321,3.847321,0.0,0.000000,...,0.000000,0.000000,3.847321,4.529743,0.0,0.0,0.00000,3.847321,3.847321,0.0
ba7f733a4f75,0.0,0.0,0.0,0.0,0.000000,3.436846,3.436846,0.000000,0.0,4.513782,...,3.436846,0.000000,4.113780,5.020215,0.0,0.0,0.00000,3.436846,4.113780,0.0
fbcf2443ffb2,0.0,0.0,0.0,0.0,0.000000,0.000000,4.196826,0.000000,0.0,0.000000,...,0.000000,4.196826,4.196826,4.196826,0.0,0.0,3.51861,4.196826,3.518610,0.0


In [9]:
# Subset to keep important columns and turn them into a numpy array
X0 = X[important_cols].values

# Clean up memory
del X
gc.collect()

# Read test and convert to sparse matrix
Xt = pd.read_hdf(FP_CITE_TEST_INPUTS).drop(columns = constant_cols)
cell_index_test = Xt.index
meta_test = metadata_df.reindex(cell_index_test)

# Subset to keep important columns
X0t = Xt[important_cols].values

# Clean up memory
del Xt
gc.collect()

# Standardize the data on X0, note that we use the same scaler for X0t
st = StandardScaler()
X0 = st.fit_transform(X0)
X0t = st.transform(X0t)

print(f'X0 shape {X0.shape} X0t shape {X0t.shape}')

X0 shape (70988, 84) X0t shape (48663, 84)


### Upload train/test files already reduced (TruncatedSVD)

In [ ]:
# Boolean flag: if True, compute & save SVD; if False, load precomputed arrays
RUN_SVD = False

PROCESSED_DIR = DATA_DIR / "processed"
TRAIN_PKL = PROCESSED_DIR / "train_Citeseq_truncated_512.pkl"
TEST_PKL  = PROCESSED_DIR / "test_Citeseq_truncated_512.pkl"

if RUN_SVD:
    print("Computing SVD embeddings...")
    # Dimension of SVD
    n_components = 512
    svd = TruncatedSVD(n_components=n_components, random_state=42)

    # 1. Load raw, drop constant cols
    X_raw  = pd.read_hdf(DATA_DIR / "raw" / "train_cite_inputs.h5").drop(columns=constant_cols)
    Xt_raw = pd.read_hdf(DATA_DIR / "raw" / "test_cite_inputs.h5").drop(columns=constant_cols)

    # 2. Fit SVD on training, transform both
    X = svd.fit_transform(X_raw)
    Xt  = svd.transform(Xt_raw)

    # 3. Persist to disk
    PROCESSED_DIR.mkdir(exist_ok=True)
    with open(TRAIN_PKL, "wb") as f:
        pickle.dump(X, f)
    with open(TEST_PKL, "wb") as f:
        pickle.dump(Xt, f)

    # 4. Cleanup
    del X_raw, Xt_raw
    gc.collect()

    print(f"Computed and saved SVD embeddings X (512 PCs): {X.shape}, X_test: {Xt.shape}")


else:
    # Load precomputed embeddings
    with open(TRAIN_PKL, "rb") as f:
        X = pickle.load(f)
    with open(TEST_PKL, "rb") as f:
        Xt = pickle.load(f)

    print(f"Loaded precomputed SVD embeddings X (512 PCs): {X.shape}, X_test: {Xt.shape}")


Loaded precomputed SVD embeddings X (512 PCs): (70988, 512), X_test: (48663, 512)


### Target normalization

##### Our target is the protein expression levels for each cell in the training set
##### We will predict protein expression from RNA expression embeddings

In [11]:
# Load targets, our target is the protein expression levels for each cell in the training set
print(f'Reading CITE-seq targets from {FP_CITE_TRAIN_TARGETS}')
# Note: This file contains the protein expression levels for each cell in the training set
Y = pd.read_hdf(FP_CITE_TRAIN_TARGETS)

Y.head()

Reading CITE-seq targets from /share/crsp/lab/pkaiser/ddlin/single-cell-multimodal-ml/data/raw/train_cite_targets.h5


gene_id,CD86,CD274,CD270,CD155,CD112,CD47,CD48,CD40,CD154,CD52,...,CD94,CD162,CD85j,CD23,CD328,HLA-E,CD82,CD101,CD88,CD224
cell_id,,,,,,,,,,,,,,,,,,,,,
45006fe3e4c8,1.167804,0.622530,0.106959,0.324989,3.331674,6.426002,1.480766,-0.728392,-0.468851,-0.073285,...,-0.448390,3.220174,-0.533004,0.674956,-0.006187,0.682148,1.398105,0.414292,1.780314,0.548070
d02759a80ba2,0.818970,0.506009,1.078682,6.848758,3.524885,5.279456,4.930438,2.069372,0.333652,-0.468088,...,0.323613,8.407108,0.131301,0.047607,-0.243628,0.547864,1.832587,0.982308,2.736507,2.184063
c016c6b0efa5,-0.356703,-0.422261,-0.824493,1.137495,0.518924,7.221962,-0.375034,1.738071,0.142919,-0.971460,...,1.348692,4.888579,-0.279483,-0.131097,-0.177604,-0.689188,9.013709,-1.182975,3.958148,2.868600
ba7f733a4f75,-1.201507,0.149115,2.022468,6.021595,7.258670,2.792436,21.708519,-0.137913,1.649969,-0.754680,...,1.504426,12.391979,0.511394,0.587863,-0.752638,1.714851,3.893782,1.799661,1.537249,4.407671
fbcf2443ffb2,-0.100404,0.697461,0.625836,-0.298404,1.369898,3.254521,-1.659380,0.643531,0.902710,1.291877,...,0.777023,6.496499,0.279898,-0.841950,-0.869419,0.675092,5.259685,-0.835379,9.631781,1.765445


In [12]:
# Normalize Y by subtracting the mean and dividing by the standard deviation for each cell
# This is important for training stability and performance
Y = Y.values
Y -= Y.mean(axis=1).reshape(-1, 1)
Y /= Y.std(axis=1).reshape(-1, 1)
Y.shape

(70988, 140)

### Let's keep only some features

In [13]:
# Concatenate the 75 SVD embeddings and the X0 with selected cell surface proteins
# These are information from RNA Expression
X = np.hstack((X[:,:75],X0))
X.shape

(70988, 159)

### Tensorflow Keras librairies

In [14]:
import math

import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import ReduceLROnPlateau, LearningRateScheduler, EarlyStopping
from tensorflow.keras.layers import Dense, Input, Concatenate, Dropout, BatchNormalization

# Check if GPU is available
if tf.config.list_physical_devices('GPU'):
    print("GPU is available")
else:
    print("GPU is not available, using CPU")

2025-07-22 18:53:53.876489: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-22 18:53:54.439562: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-22 18:53:54.602645: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753235634.721339 3305803 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753235634.741439 3305803 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753235635.241123 3305803 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

GPU is not available, using CPU


2025-07-22 18:54:02.545338: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


### Metric and loss function

In [15]:
def correlation_score(y_true, y_pred):
    """
    Compute the mean Pearson correlation coefficient between true and predicted values.

    Parameters:
    - y_true: Ground truth values (numpy array or pandas DataFrame) of shape (n_samples, n_targets).
    - y_pred: Predicted values (numpy array or pandas DataFrame) of shape (n_samples, n_targets).

    Returns:
    - float: Mean Pearson correlation coefficient across all samples.
    """
    # Convert pandas DataFrame to numpy array if needed
    if type(y_true) == pd.DataFrame:
        y_true = y_true.values
    if type(y_pred) == pd.DataFrame:
        y_pred = y_pred.values
    
    corrsum = 0  # Initialize sum of correlation coefficients
    # Iterate over each sample (row) to compute correlation
    for i in range(len(y_true)):
        # Calculate Pearson correlation coefficient for the i-th sample
        # np.corrcoef returns a 2x2 matrix; [1,0] is the off-diagonal (true vs. pred)
        corrsum += np.corrcoef(y_true[i], y_pred[i])[1, 0]
    
    # Return the mean correlation across all samples
    return corrsum / len(y_true)

def negative_correlation_loss(y_true, y_pred):
    """
    Custom Keras loss function to compute the negative mean Pearson correlation coefficient.

    Parameters:
    - y_true: Ground truth tensor of shape (batch_size, n_targets).
    - y_pred: Predicted tensor of shape (batch_size, n_targets).

    Returns:
    - Tensor: Negative mean Pearson correlation coefficient (to be minimized).
    """
    # Compute mean of predictions across features (axis=1) for each sample
    my = K.mean(tf.convert_to_tensor(y_pred), axis=1)
    
    # Reshape mean to (batch_size, 1) and tile to match y_true shape (batch_size, n_targets)
    my = tf.tile(tf.expand_dims(my, axis=1), (1, y_true.shape[1]))
    
    # Center predictions by subtracting the mean (y_pred - mean(y_pred))
    ym = y_pred - my
    
    # Numerator: Sum of element-wise product of true and centered predicted values
    r_num = K.sum(tf.multiply(y_true, ym), axis=1)
    
    # Denominator: Product of standard deviation of predictions and sqrt(n_targets)
    r_den = tf.sqrt(K.sum(K.square(ym), axis=1) * float(y_true.shape[-1]))
    
    # Compute mean Pearson correlation coefficient across the batch
    r = tf.reduce_mean(r_num / r_den)
    
    # Return negative correlation (to minimize as a loss)
    return -r

### Model and parameters

##### A simple “wide‑and‑deep” MLP

In [16]:
LR_START = 0.01
BATCH_SIZE = 512

def create_model():
    # regularizers & dropout rate
    reg1 = 9.613e-06
    reg2 = 1e-07
    REG1 = tf.keras.regularizers.l2(reg1)
    REG2 = tf.keras.regularizers.l2(reg2)
    DROP = 0.1

    activation = 'selu'
    inputs = Input(shape=(X.shape[1],))

    # four successive Dense→Dropout blocks
    x0 = Dense(256, kernel_regularizer=REG1, activation=activation)(inputs)
    x0 = Dropout(DROP)(x0)

    x1 = Dense(512, kernel_regularizer=REG1, activation=activation)(x0)
    x1 = Dropout(DROP)(x1)

    x2 = Dense(512, kernel_regularizer=REG1, activation=activation)(x1)
    x2 = Dropout(DROP)(x2)

    x3 = Dense(Y.shape[1], kernel_regularizer=REG1, activation=activation)(x2)
    x3 = Dropout(DROP)(x3)

    # concatenate all four intermediate outputs
    x = Concatenate()([x0, x1, x2, x3])

    # final linear layer to produce Y.shape[1] outputs
    x = Dense(Y.shape[1], kernel_regularizer=REG2, activation='linear')(x)

    return Model(inputs, x)

test_model = create_model()
test_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 159)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │     40,960 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 512)       │    131,584 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 512)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 512)       │    262,656 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 512)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 140)       │     71,820 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 140)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1420)      │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_1[0][0],  │
│                     │                   │            │ dropout_2[0][0],  │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 140)       │    198,940 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 705,960 (2.69 MB)

 Trainable params: 705,960 (2.69 MB)

 Non-trainable params: 0 (0.00 B)

### Training

-  The correlation scores (~0.89 per fold and ~0.89 for out of fold) suggest the model is performing well for using the embedded RNA expression information to predict protein expression.

In [17]:
%%time

# Create the submissions directory (and parents if needed)
model_dir = DATA_DIR.joinpath("models", "citeseq", "submissions")
model_dir.mkdir(parents=True, exist_ok=True)

# Also create the directory for weights if it might not exist
weights_dir = DATA_DIR.joinpath("models", "citeseq")
weights_dir.mkdir(parents=True, exist_ok=True)

EPOCHS = 300 
N_SPLITS = 3

pred_train = np.zeros((Y.shape[0],Y.shape[1]))

np.random.seed(1)
tf.random.set_seed(1)
score_list = []
kf = GroupKFold(n_splits=N_SPLITS)
score_list = []

# GroupKFold to ensure that cells from the same donor are not in both train and validation sets, preventing data leakage
# This way, we can ensure that the model generalizes well to unseen data
for fold, (idx_tr, idx_va) in enumerate(kf.split(X, groups=meta.donor)):
    start_time = datetime.datetime.now()
    model = None
    gc.collect()
    
    X_tr = X[idx_tr]
    y_tr = Y[idx_tr]
    X_va = X[idx_va]
    y_va = Y[idx_va]

    lr = ReduceLROnPlateau(
                    monitor = "val_loss",
                    factor = 0.9, 
                    patience = 4, 
                    verbose = VERBOSE)

    es = EarlyStopping(
                    monitor = "val_loss",
                    patience = 40, 
                    verbose = VERBOSE,
                    mode = "min", 
                    restore_best_weights = True)

    model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
                    filepath = weights_dir.joinpath("citeseq.weights.h5"),
                    save_weights_only = True,
                    monitor = 'val_loss',
                    mode = 'min',
                    save_best_only = True)

    callbacks = [
                    lr, 
                    es, 
                    model_checkpoint_callback
                    ]
    
    model = create_model()
    
    model.compile(
                optimizer = tf.keras.optimizers.Adam(learning_rate=LR_START),
                metrics = [negative_correlation_loss],
                loss = negative_correlation_loss
                 )
    # Training
    model.fit(
                X_tr,
                y_tr, 
                validation_data=(
                                X_va,
                                y_va), 
                epochs = EPOCHS,
                verbose = VERBOSE,
                batch_size = BATCH_SIZE,
                shuffle = True,
                callbacks = callbacks)

    del X_tr, y_tr 
    gc.collect()
    
    model.load_weights(weights_dir.joinpath("citeseq.weights.h5"))
    model.save(model_dir.joinpath(f"model_{fold}.keras"))
    print('model saved')
    
    #  Model validation
    y_va_pred = model.predict(X_va)
    corrscore = correlation_score(y_va, y_va_pred)
    pred_train[idx_va] = y_va_pred
    
    print(f"Fold {fold}, correlation =  {corrscore:.5f}")
    del X_va, y_va, y_va_pred
    gc.collect()
    score_list.append(corrscore)

# Show overall score
print(f"{Fore.GREEN}{Style.BRIGHT}Mean correlation = {np.array(score_list).mean():.5f}{Style.RESET_ALL}")
score_total = correlation_score(Y, pred_train)
print(f"{Fore.BLUE}{Style.BRIGHT}Out of fold correlation = {score_total:.5f}{Style.RESET_ALL}")

model saved
776/776 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Fold 0, correlation =  0.89081
model saved
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Fold 1, correlation =  0.89672
model saved
694/694 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Fold 2, correlation =  0.89311
Mean correlation = 0.89355
Out of fold correlation = 0.89353
CPU times: user 1h 56min 21s, sys: 8min 18s, total: 2h 4min 40s
Wall time: 15min 26s


## ------ Sparse Data Generation and Dimension Reduction---------
- scATAC-seq multiome data is highly sparse (>95% empty) and noisy.
- We will address this by converting the .h5 output to a sparse matrix and applying dimension reduction.
- These preprocessing steps are essential for downstream machine learning tasks.
- Implementation details are in the dimension-reduction notebook.

## ------ Multiome MODEL ---------

In [22]:
# Paths for saving/loading the reduced arrays and SVD objects (using "embeddings" and "model" for clarity)
TRAIN_INPUTS_EMBEDDINGS_PKL = PROCESSED_DIR / "train_multi_inputs_embeddings_512.pkl"
TEST_INPUTS_EMBEDDINGS_PKL = PROCESSED_DIR / "test_multi_inputs_embeddings_512.pkl"
TRAIN_TARGETS_EMBEDDINGS_PKL = PROCESSED_DIR / "train_multi_targets_embeddings_512.pkl"
SVD_INPUTS_MODEL_PKL = PROCESSED_DIR / "svd_inputs_model_512.pkl"
SVD_TARGETS_MODEL_PKL = PROCESSED_DIR / "svd_targets_model_512.pkl"

# Load train inputs embeddings
with open(TRAIN_INPUTS_EMBEDDINGS_PKL, "rb") as f:
    X = pickle.load(f)
print(f"Loaded X: {X.shape}")

# Load test inputs embeddings
with open(TEST_INPUTS_EMBEDDINGS_PKL, "rb") as f:
    Xt = pickle.load(f)
print(f"Loaded Xt: {Xt.shape}")

# Load train targets embeddings
with open(TRAIN_TARGETS_EMBEDDINGS_PKL, "rb") as f:
    Y = pickle.load(f)
print(f"Loaded Y: {Y.shape}")

# Load SVD inputs model
with open(SVD_INPUTS_MODEL_PKL, "rb") as f:
    svd_inputs = pickle.load(f)
print("Loaded svd_inputs")

# Load SVD targets model
with open(SVD_TARGETS_MODEL_PKL, "rb") as f:
    svd_targets = pickle.load(f)
print("Loaded svd_targets")

Loaded X: (105942, 512)
Loaded Xt: (55935, 512)
Loaded Y: (105942, 512)
Loaded svd_inputs
Loaded svd_targets


In [ ]:
# Regarding INDEX_train_multiome: No need to save it separately, as it can be loaded directly from the metadata NPZ file when needed, e.g.:
# metadata = np.load(PROCESSED_DIR / "train_multi_inputs_metadata.npz", allow_pickle=True)
# INDEX_train_multiome = metadata['index']

# Now you can use X[:, :40] and Y for training the NN as in your example
# For prediction, use Xt[:, :40] as input to the model, then reconstruct with pred @ svd_targets.components_
# Metadata (index, columns) can still be loaded from the original npz files as needed

## Upload of the multiome files after TruncatedSVD

In [14]:
with open('../input/targets-multiome-sparse-scaled/INDEX_train_multiome.pkl','rb') as f: INDEX_train_multiome = pickle.load(f)
with open('../input/targets-multiome-sparse-scaled/train_512.pkl','rb') as f: X = pickle.load(f)
with open('../input/targets-multiome-sparse-scaled/pca_train_512.pkl','rb') as f: pca_train = pickle.load(f)
with open('../input/targets-multiome-sparse-scaled/pca_target_512.pkl','rb') as f: pca_target = pickle.load(f)
with open('../input/targets-multiome-sparse-scaled/Y_512.pkl','rb') as f: Y = pickle.load(f)

In [15]:
X = X[:,:40]
X.shape

(105942, 40)

In [16]:
metadata_df = pd.read_csv('../input/open-problems-multimodal/metadata.csv',index_col='cell_id')
metadata_df = metadata_df[metadata_df.technology=="multiome"]
meta = metadata_df.reindex(INDEX_train_multiome)

In [17]:
Y.shape, X.shape

((105942, 512), (105942, 40))

## Training for Multiome

In [18]:
import warnings
warnings.filterwarnings("ignore")

N_SPLIT = 3
#kf = KFold(n_splits=N_SPLIT, shuffle=True, random_state=42)
kf = GroupKFold(n_splits = 3)

for fold,(idx_tr, idx_va) in enumerate(kf.split(X,groups=meta.donor)):
    
    X_tr = X[idx_tr]
    y_tr = Y[idx_tr]
    
    X_va = X[idx_va]
    y_va = Y[idx_va] 
    
    model = create_model()
    
    lr = ReduceLROnPlateau(
                monitor = "val_loss",
                factor = 0.9, 
                patience = 4, 
                verbose = VERBOSE)
    
    es = EarlyStopping(
                monitor = "val_loss",
                patience = 30, 
                verbose = VERBOSE,
                mode = "min", 
                restore_best_weights = True)

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss = 'mse',
                  metrics=None)
    model.fit(X_tr,
              y_tr,
              validation_data=(X_va,y_va),
              epochs =500,
              verbose = VERBOSE,
              batch_size=256,
              callbacks = [es,lr]
             )
    pred = model.predict(X_va)
    
    print(f'\n --------- FOLD {fold} -----------')
    print(f'Mean squared error = {np.round(mean_squared_error(y_va,pred),2)}')
   
    filename = f"model_{fold}"
    model.save(filename)
    print('model saved :',filename)
        
    del X_tr,X_va,y_tr,y_va
    gc.collect()


 --------- FOLD 0 -----------
Mean squared error = 2.609999895095825
model saved : model_0

 --------- FOLD 1 -----------
Mean squared error = 2.490000009536743
model saved : model_1

 --------- FOLD 2 -----------
Mean squared error = 2.5399999618530273
model saved : model_2


## Test predictions for Multiome

In [19]:
multi_test_x = scipy.sparse.load_npz("../input/multimodal-single-cell-as-sparse-matrix/test_multi_inputs_values.sparse.npz")
multi_test_x = pca_train.transform(multi_test_x)
multi_test_x = multi_test_x[:,:40]
multi_test_x.shape

(55935, 40)

In [20]:
preds = np.zeros((multi_test_x.shape[0], 23418), dtype='float16')

for fold in range(N_SPLIT):
    print(f'fold {fold} prediction')
    model = tf.keras.models.load_model(f"model_{fold}")
    preds += (model.predict(multi_test_x)@pca_target.components_)/N_SPLIT

    gc.collect()

fold 0 prediction
fold 1 prediction
fold 2 prediction


In [21]:
eval_ids = pd.read_parquet("../input/multimodal-single-cell-as-sparse-matrix/evaluation.parquet")
eval_ids.cell_id = eval_ids.cell_id.astype(pd.CategoricalDtype())
eval_ids.gene_id = eval_ids.gene_id.astype(pd.CategoricalDtype())

submission = pd.Series(name='target',
                       index=pd.MultiIndex.from_frame(eval_ids), 
                       dtype=np.float32)
submission

row_id    cell_id       gene_id        
0         c2150f55becb  CD86              NaN
1         c2150f55becb  CD274             NaN
2         c2150f55becb  CD270             NaN
3         c2150f55becb  CD155             NaN
4         c2150f55becb  CD112             NaN
                                           ..
65744175  2c53aa67933d  ENSG00000134419   NaN
65744176  2c53aa67933d  ENSG00000186862   NaN
65744177  2c53aa67933d  ENSG00000170959   NaN
65744178  2c53aa67933d  ENSG00000107874   NaN
65744179  2c53aa67933d  ENSG00000166012   NaN
Name: target, Length: 65744180, dtype: float32

In [22]:
y_columns = np.load("../input/multimodal-single-cell-as-sparse-matrix/train_multi_targets_idxcol.npz",
                   allow_pickle=True)["columns"]

test_index = np.load("../input/multimodal-single-cell-as-sparse-matrix/test_multi_inputs_idxcol.npz",
                    allow_pickle=True)["index"]

cell_dict = dict((k,v) for v,k in enumerate(test_index)) 
assert len(cell_dict)  == len(test_index)

gene_dict = dict((k,v) for v,k in enumerate(y_columns))
assert len(gene_dict) == len(y_columns)

eval_ids_cell_num = eval_ids.cell_id.apply(lambda x:cell_dict.get(x, -1))
eval_ids_gene_num = eval_ids.gene_id.apply(lambda x:gene_dict.get(x, -1))
valid_multi_rows = (eval_ids_gene_num !=-1) & (eval_ids_cell_num!=-1)

submission.iloc[valid_multi_rows] = preds[eval_ids_cell_num[valid_multi_rows].to_numpy(),
eval_ids_gene_num[valid_multi_rows].to_numpy()]

del eval_ids_cell_num, eval_ids_gene_num, valid_multi_rows, eval_ids, test_index, y_columns
gc.collect()

submission

row_id    cell_id       gene_id        
0         c2150f55becb  CD86                    NaN
1         c2150f55becb  CD274                   NaN
2         c2150f55becb  CD270                   NaN
3         c2150f55becb  CD155                   NaN
4         c2150f55becb  CD112                   NaN
                                             ...   
65744175  2c53aa67933d  ENSG00000134419    2.785156
65744176  2c53aa67933d  ENSG00000186862   -0.379150
65744177  2c53aa67933d  ENSG00000170959   -0.375732
65744178  2c53aa67933d  ENSG00000107874    0.167603
65744179  2c53aa67933d  ENSG00000166012    2.480469
Name: target, Length: 65744180, dtype: float32

## Total submission

In [23]:
submission.reset_index(drop=True, inplace=True)
submission.index.name = 'row_id'

cite_submission = pd.read_csv("submission_lolo_1.csv")
cite_submission = cite_submission.set_index("row_id")
cite_submission = cite_submission["target"]
submission[submission.isnull()] = cite_submission[submission.isnull()]
submission
# == > score 0.812


row_id
0           0.094605
1          -0.162362
2          -0.405332
3          -0.302582
4           1.114355
              ...   
65744175    2.785156
65744176   -0.379150
65744177   -0.375732
65744178    0.167603
65744179    2.480469
Name: target, Length: 65744180, dtype: float32

In [24]:
sub_ensembling = pd.read_csv('../input/5-5-msci22-ensembling-citeseq/submission.csv')
submission1 = sub_ensembling.copy()
submission1['target'] = 0.4 * submission + 0.6 * sub_ensembling['target']
submission1

,row_id,target
0,0,0.182096
1,1,0.010811
2,2,-0.101281
3,3,1.442052
4,4,2.616124
...,...,...
65744175,65744175,5.422715
65744176,65744176,-0.302524
65744177,65744177,-0.293852
65744178,65744178,0.755234


In [25]:
submission1.to_csv("submission_lolo_total_ensembling.csv", index = False)